# 🦉 Strixlibe × Google Colab G4 GPU (Blackwell 96GB) LLM Server
自律型ペンテストエージェント「Strixlibe」向けに、Google Colab の最新 **G4 GPU (NVIDIA RTX PRO 6000 Blackwell 96GB VRAM)** 上で、ガードレールフリーな最高峰オープンソースモデル（**Qwen3.8-27B**、DeepSeek-R1-32B フル精度、Qwen2.5-72B、Llama-3.3-70B 等）を vLLM でホストし、Cloudflare Tunnel 経由で安全に連携するワンクリック・ノートブックです。

### ⚡ G4 (Blackwell 96GB) の特長
- **Qwen3.8-27B フル対応**: 最新の **Qwen3.8-27B** を BF16 フル精度（VRAM 約54GB）で余裕でホスト可能
- **圧倒的 VRAM**: 約 96GB GDDR7 により、32B/27B モデルの BF16 フル精度（量子化劣化なし）や 72B クラスのモデルが稼働
- **省CU・低コスト**: 約 6.4 〜 8.7 CU/h（A100 より約30%安く、月100CU枠で約12〜15時間稼働可能）
- **自動トンネリング**: Cloudflare Quick Tunnel でアカウント不要・即時URL発行
- **自動同期**: Strixlibe Dispatcher に接続先を自動登録（Colab 停止時は OpenCode(Zen) へ自動フォールバック）
- **CU保護**: 20分間リクエストが途絶えると自動でランタイムを切断・停止

### 1. 設定項目（フォーム）

In [ ]:
#@title 🚀 サーバー基本設定 { run: "auto" }
MODEL_NAME = "Qwen/Qwen3.8-27B" #@param ["Qwen/Qwen3.8-27B", "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B", "Qwen/Qwen2.5-72B-Instruct-AWQ", "casperhansen/llama-3.3-70b-instruct-awq", "casperhansen/deepseek-r1-distill-qwen-32b-awq", "Qwen/Qwen2.5-Coder-32B-Instruct"]
MAX_MODEL_LEN = 32768 #@param [16384, 32768, 65536] {type:"raw"}
DISPATCHER_URL = "https://strix-dispatcher-2acg5ejyka-uw.a.run.app" #@param {type:"string"}
DISPATCHER_API_KEY = "" #@param {type:"string"}
SLACK_WEBHOOK_URL = "" #@param {type:"string"}
IDLE_TIMEOUT_MINUTES = 20 #@param {type:"integer"}
TARGET_ENV = "all" #@param ["all", "staging", "production"]

### 2. GPU確認とパッケージセットアップ

In [ ]:
# GPUの確認（G4: RTX PRO 6000 96GB GDDR7）
!nvidia-smi

# 依存パッケージの導入（Colab標準パッケージとの警告を無視して安全にインストール）
!pip install -q --no-warn-conflicts vllm hf_transfer || true

# cloudflared のダウンロード
!curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# vLLM の動作確認
try:
    import vllm
    print(f"\n[+] vLLM {vllm.__version__} is successfully loaded and ready!")
except Exception as e:
    print(f"\n[!] vLLM verification notice: {e}")

### 3. vLLM サーバーと Cloudflare Tunnel の起動

In [ ]:
import subprocess
import time
import re
import requests
import threading
import json

# vLLM サーバーの起動コマンド（96GB VRAM を活用し最大 92% メモリ確保）
print(f"[*] Starting vLLM with model: {MODEL_NAME} (Max Context: {MAX_MODEL_LEN})...")
vllm_cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_NAME,
    "--port", "8000",
    "--host", "0.0.0.0",
    "--trust-remote-code",
    "--dtype", "auto",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", "0.92"
]

vllm_proc = subprocess.Popen(vllm_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# vLLM のヘルスチェック待機（モデルDLと初期化）
print("[*] Waiting for vLLM to initialize (approx 2-5 minutes for model download & weights loading)...")
while True:
    try:
        r = requests.get("http://localhost:8000/v1/models", timeout=2)
        if r.status_code == 200:
            print("[+] vLLM is healthy and ready to serve!")
            break
    except:
        pass
    time.sleep(5)

# Cloudflare Quick Tunnel の起動
print("[*] Establishing Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

tunnel_url = None
while True:
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        print(f"\n🎉 ==========================================================")
        print(f"[+] Cloudflare Tunnel established successfully!")
        print(f"[+] Public URL: {tunnel_url}")
        print(f"[+] OpenAI API Base: {tunnel_url}/v1")
        print(f"==========================================================\n")
        break

api_base = f"{tunnel_url}/v1"

### 4. 疎通テスト（トンネル経由での推論テスト）

In [ ]:
print(f"[*] Sending test prompt to: {api_base}/chat/completions ...")
test_payload = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "user", "content": "Strixペンテストエージェントからの疎通確認テストです。接続が正常であることを、日本語で1〜2文で応答してください。"}
    ],
    "max_tokens": 150,
    "temperature": 0.6
}

try:
    resp = requests.post(f"{api_base}/chat/completions", json=test_payload, timeout=40)
    if resp.status_code == 200:
        result = resp.json()
        reply = result["choices"][0]["message"]["content"]
        print("\n[+] 推論テスト成功！モデルからの返答:")
        print("--------------------------------------------------")
        print(reply.strip())
        print("--------------------------------------------------")
    else:
        print(f"[-] テスト失敗: HTTP {resp.status_code} - {resp.text}")
except Exception as e:
    print(f"[-] テスト例外: {e}")

### 5. Dispatcher 自動同期 & Slack 通知

In [ ]:
# Dispatcher へのセッション登録
if DISPATCHER_URL and DISPATCHER_API_KEY:
    try:
        reg_payload = {
            "provider": "colab-vllm",
            "base_url": api_base,
            "model_name": MODEL_NAME,
            "env_target": TARGET_ENV,
            "ttl_minutes": 120
        }
        headers = {"X-API-Key": DISPATCHER_API_KEY, "Content-Type": "application/json"}
        reg_res = requests.post(f"{DISPATCHER_URL}/api/v1/llm-session", json=reg_payload, headers=headers, timeout=10)
        print(f"[+] Registered to Dispatcher: status {reg_res.status_code}")
    except Exception as e:
        print(f"[-] Dispatcher registration skipped or failed: {e}")
else:
    print("[*] Dispatcher 未設定のため同期をスキップしました（手動接続可能モード）。")

# Slack 通知
if SLACK_WEBHOOK_URL:
    try:
        msg = (
            f"🚀 *Strixlibe Colab G4 GPU LLM Server is ONLINE!*\n"
            f"• *GPU*: G4 (RTX PRO 6000 Blackwell 96GB)\n"
            f"• *Model*: `{MODEL_NAME}`\n"
            f"• *Endpoint*: `{api_base}`\n"
            f"• *Env*: `{TARGET_ENV}`\n"
            f"• *Watchdog*: {IDLE_TIMEOUT_MINUTES} mins auto-kill\n"
            f"💡 Strixlibe will automatically route scans to this model."
        )
        requests.post(SLACK_WEBHOOK_URL, json={"text": msg}, timeout=5)
        print("[+] Slack notification sent.")
    except Exception as e:
        print(f"[-] Slack notification failed: {e}")

### 6. CU保護ウォッチドッグ（バックグラウンド監視・自動シャットダウン）

In [ ]:
from google.colab import runtime

def watchdog_worker():
    print(f"[*] Watchdog background worker started. Auto-terminate after {IDLE_TIMEOUT_MINUTES}m inactivity.")
    last_seen_time = time.time()
    idle_timeout_seconds = IDLE_TIMEOUT_MINUTES * 60
    
    while True:
        time.sleep(30)
        elapsed = time.time() - last_seen_time
        if elapsed > idle_timeout_seconds:
            print(f"[!] Inactivity threshold ({IDLE_TIMEOUT_MINUTES} min) exceeded. Terminating runtime to preserve Compute Units.")
            if SLACK_WEBHOOK_URL:
                try:
                    requests.post(SLACK_WEBHOOK_URL, json={"text": "🛑 *Colab G4 LLM Server disconnected* (Idle timeout). Auto-fallback to OpenCode(Zen) engaged."}, timeout=5)
                except:
                    pass
            runtime.unassign()
            break

# バックグラウンドスレッドとして起動
t = threading.Thread(target=watchdog_worker, daemon=True)
t.start()
print("[OK] All systems operational! Tunnel is active and watchdog is monitoring.")